# 97 — GNN Bucket Action Selector

Heterogeneous GNN (tripartite: planet ↔ action ↔ planet) that selects attacks and
ship counts, trained by imitating the 90-Simulate heuristic.
Output: [[src_id, dst_id, eta, ships_to_send], ...]

In [ ]:
%run 96-library.py
%run 90-Simulate10Next_Conqueror2_Supplier_prod_per_step.py
%run 97-library.py
import torch
import torch.nn as nn
from torch_geometric.nn import SAGEConv
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML, display
import math, copy

In [ ]:
_COLORS = {0: 'steelblue', 1: 'tomato', -1: '#888888'}

def make_animation(snapshots, title='', interval=150):
    fig, ax = plt.subplots(figsize=(6, 6))
    fig.patch.set_facecolor('#111122')
    def draw(frame):
        snap = snapshots[frame]
        ax.cla()
        ax.set_xlim(0, 100); ax.set_ylim(100, 0)
        ax.set_aspect('equal'); ax.set_facecolor('#111122')
        ax.tick_params(colors='#aaaaaa')
        for sp in ax.spines.values(): sp.set_edgecolor('#444444')
        ax.set_title(f"{title}  (step {snap['step']})", color='white', fontsize=11)
        ax.add_patch(plt.Circle((50, 50), 10, color='gold', zorder=2, alpha=0.9))
        for p in snap['planets']:
            pid, owner, x, y, radius, ships, production = p
            c = _COLORS.get(owner, '#888888')
            ax.add_patch(plt.Circle((x, y), radius, color=c, alpha=0.85, zorder=3))
            ax.text(x, y,   str(ships),         ha='center', va='center', color='white', fontsize=7, fontweight='bold', zorder=4)
            ax.text(x, y+2, str(pid),            ha='center', va='center', color='red',   fontsize=7, fontweight='bold', zorder=4)
            ax.text(x, y-2, '+'+str(production), ha='center', va='center', color='white', fontsize=5, fontweight='bold', zorder=4)
        return []
    ani = animation.FuncAnimation(fig, draw, frames=len(snapshots), interval=interval)
    plt.close()
    return HTML(ani.to_jshtml())

In [ ]:
data0, snaps0, ai0 = generate_sample_97(42)
n_act = data0['action'].x.shape[0]
n_pos = int(data0['action'].y.sum().item())
print(f"Planets: {data0['planet'].x.shape[0]}")
print(f"Action nodes: {n_act}  |  Positive (heuristic selected): {n_pos}")
print(f"Spawns edges: {data0['planet','spawns','action'].edge_index.shape[1]}")
print(f"Attacks edges: {data0['action','attacks','planet'].edge_index.shape[1]}")
make_animation(snaps0, title='Sample seed=42', interval=200)

In [ ]:
print("Generating train dataset (1000 samples)...")
train_dataset = [generate_sample_97(i) for i in range(1000)]
print("Generating test dataset (20 samples)...")
test_dataset  = [generate_sample_97(10000 + i) for i in range(20)]

train_pos = sum(int(d.y.sum()) for d, _, _ in train_dataset)
train_tot = sum(d['action'].x.shape[0] for d, _, _ in train_dataset)
print(f"Train: {train_tot} action nodes total, {train_pos} positive ({100*train_pos/max(train_tot,1):.1f}%)")